# 03 — Facteurs économiques & importance SHAP

**Question centrale :** Peut-on résumer 249 features brutes en un ensemble de facteurs interprétables ?

## Pourquoi des facteurs plutôt que des features brutes ?

On part de ~249 variables brutes. Les problèmes avec ça :
1. **Multicolinéarité** — WASDE exports et WASDE crush sont très corrélés
2. **Interprétation impossible** — `f_raw__cot_pm_long_chg4` ne dit rien
3. **Instabilité** — une petite variation d'une colonne USDA peut redistribuer toute l'importance
4. **Leakage latent** — une variable brute peut embarquer de l'info future si mal alignée

**Solution :** créer des facteurs économiques = agrégations interprétables de variables brutes.

| Facteur | Famille | Signal économique |
|---|---|---|
| `factor_wasde_supply_demand` | Fondamental | Bilan offre/demande USDA |
| `factor_market_momentum` | Technique | Tendance récente du prix |
| `factor_market_volatility` | Risque | Régime de volatilité |
| `factor_cot_positioning` | Sentiment | Positions nettes spéculateurs |
| `factor_seasonality` | Calendrier | Position dans le cycle annuel |
| `factor_weather_belt_stress` | Météo | Stress thermique Corn Belt |
| `factor_crop_condition` | Production | Qualité culturale NASS |
| `factor_ethanol_demand` | Demande | Production éthanol EIA |
| `factor_cross_commodity` | Macro | Ratio maïs/blé/soja/DXY |

In [ ]:
import sys
sys.path.insert(0, '../../src')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path('../../')
plt.rcParams['figure.figsize'] = (14, 6)
plt.style.use('seaborn-v0_8-whitegrid')

# Chargement
factors = pd.read_parquet(ROOT / 'data/processed/factors.parquet')
factors['Date'] = pd.to_datetime(factors['Date'])

shap_imp = pd.read_parquet(ROOT / 'artefacts/professional_study/shap_importance.parquet')
fam_imp  = pd.read_parquet(ROOT / 'artefacts/professional_study/family_importance.parquet')

factor_cols = [c for c in factors.columns if c.startswith('factor_')]
print(f'Facteurs disponibles ({len(factor_cols)}) :')
for c in factor_cols:
    nn = factors[c].notna().sum()
    print(f'  {c:45s}  {nn:5d} obs non-null ({nn/len(factors):.0%})')

## 1. Évolution des facteurs dans le temps

Les facteurs sont des z-scores à expansion (anti-leakage). Ils doivent être stationnaires et lisibles.

In [ ]:
key_factors = [c for c in factor_cols if c in factors.columns][:6]

fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex=True)
for ax, col in zip(axes.flat, key_factors):
    s = factors.set_index('Date')[col].dropna()
    ax.plot(s.index, s.values, lw=0.8, color='steelblue')
    ax.axhline(0, color='gray', lw=0.5)
    ax.axhline(2, color='red', lw=0.5, ls='--', alpha=0.5)
    ax.axhline(-2, color='green', lw=0.5, ls='--', alpha=0.5)
    ax.set_title(col.replace('factor_', '').replace('_', ' ').title())
    ax.set_ylim(-4, 4)

plt.suptitle('Facteurs économiques (z-scores expansifs) — 2000-2025', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Corrélation entre facteurs

Si deux facteurs sont trop corrélés, ils portent la même information. Seuil critique : |r| > 0.70.

In [ ]:
if len(factor_cols) >= 2:
    corr = factors[factor_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax, square=True,
                xticklabels=[c.replace('factor_','') for c in factor_cols],
                yticklabels=[c.replace('factor_','') for c in factor_cols])
    ax.set_title('Corrélations entre facteurs économiques', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Paires à problème
    high = [(factor_cols[i], factor_cols[j], corr.iloc[i,j])
            for i in range(len(factor_cols)) for j in range(i+1, len(factor_cols))
            if abs(corr.iloc[i,j]) > 0.70]
    if high:
        print('Paires avec |r| > 0.70 :')
        for a,b,r in sorted(high, key=lambda x: -abs(x[2])):
            print(f'  {a} × {b}  →  r={r:.3f}')
    else:
        print('Aucune paire avec |r| > 0.70 — facteurs orthogonaux ✓')

## 3. Importance SHAP par facteur (tous horizons)

On charge les résultats SHAP calculés lors de l'étude professionnelle.

SHAP mesure la **contribution marginale** de chaque variable au modèle Random Forest.
C'est différent de la corrélation simple : il capture les interactions non-linéaires.

In [ ]:
print('Colonnes SHAP :', list(shap_imp.columns))
print('Horizons :', shap_imp['horizon'].unique() if 'horizon' in shap_imp.columns else 'N/A')
shap_imp.head(5)

In [ ]:
val_col = 'abs_coef' if 'abs_coef' in shap_imp.columns else 'mean_abs_shap'

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
for ax, h in zip(axes.flat, [5, 10, 20, 30]):
    sub = shap_imp[shap_imp['horizon'] == h].nlargest(15, val_col)
    if sub.empty:
        ax.set_title(f'H={h}j — données manquantes')
        continue
    
    name_col = 'factor' if 'factor' in sub.columns else 'feature'
    family_col = 'family' if 'family' in sub.columns else None
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(sub)))
    bars = ax.barh(range(len(sub)), sub[val_col].values, color=colors)
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels(sub[name_col].values, fontsize=9)
    ax.set_title(f'Top 15 facteurs — H={h}j', fontsize=11)
    ax.set_xlabel(val_col)
    ax.invert_yaxis()

plt.suptitle('Importance SHAP des facteurs par horizon de prédiction', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Importance par famille (SHAP agrégé)

On aggrège les scores SHAP par famille économique pour voir **quelle catégorie de signal domine**.

In [ ]:
print('Colonnes family_importance :', list(fam_imp.columns))
fam_imp.head()

In [ ]:
share_col = 'coef_share' if 'coef_share' in fam_imp.columns else 'importance_share'
fam_col   = 'family' if 'family' in fam_imp.columns else 'famille'

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for ax, h in zip(axes, [5, 10, 20, 30]):
    sub = fam_imp[fam_imp['horizon'] == h].sort_values(share_col, ascending=False)
    if sub.empty:
        ax.set_title(f'H={h}j')
        continue
    wedges, texts, autotexts = ax.pie(
        sub[share_col],
        labels=sub[fam_col],
        autopct='%1.0f%%',
        startangle=90,
        textprops={'fontsize': 8}
    )
    ax.set_title(f'H={h}j', fontsize=11)

plt.suptitle('Répartition de l\'importance par famille économique (SHAP)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Stabilité des facteurs dans le temps

Un facteur vraiment utile doit avoir une importance **stable d'une période à l'autre**.
Si son importance varie beaucoup, c'est un signal de surapprentissage ou d'instabilité structurelle.

In [ ]:
# On peut charger les benchmarks par fold pour voir la stabilité
bm = pd.read_parquet(ROOT / 'artefacts/professional_study/model_benchmarks.parquet')
print('Colonnes benchmarks :', list(bm.columns)[:15])
print('Modèles :', bm['model'].unique() if 'model' in bm.columns else 'N/A')

## 6. Ce qui manque encore dans les facteurs

### Problème actuel : la famille "others" domine

Dans les résultats SHAP actuels, beaucoup de variables sont classées `others` parce qu'elles sont des raw features qui n'ont pas encore été agrégées en facteur propre.

**Exemple :**
- `f_raw__wasde_exports` devrait être intégré dans `factor_wasde_supply_demand`
- `f_raw__corn_sma_50` devrait être `factor_market_momentum`
- `f_raw__corn_wheat_corr60` devrait être `factor_cross_commodity`

### Ce qu'il faut faire

1. Attribuer chaque `f_raw__` à une famille explicite
2. Créer un facteur pour chaque famille (agrégation pondérée)
3. Avoir `others < 10%` de l'importance totale

**→ Prochain carnet :** Benchmarks des modèles — est-ce qu'un bon jeu de facteurs bat vraiment les baselines ?